# Diagnose, Fix, Brief（診断・修正・報告）· Meridianパイロット

お客様のAIサポートエージェントが期待どおりに機能しておらず、お客様はモデルに原因があると考えています。本演習では、実際の原因を特定して修正し、お客様が経営陣に報告できる数値で修正の効果を実証していただきます。モデルの周辺は自由に変更できますが、モデル自体は変更できません。

## 開始前の確認事項

進め方は、期待どおりに動作しないお客様のシステムに、実際の案件で取り組む場合と同じです。システムを操作して挙動を観察し、仮説を立て、1か所だけ変更し、数値が変化するかを確認します。

この数値について補足します。スコアボードは、実行するたびにシステムを1回だけ動かすわけではありません。同じサポートチケットを5回処理させ、エージェントが実際に解決できた割合を報告します。1回の成功だけでは何も実証できません。エージェントの判断は実行ごとにばらつくため、個々の結果ではなく割合で判断してください。この割合と、併せて表示されるコストが評価指標です。何を変更すべきか、その変更が有効だったかは、この2つをもとに判断します。

1回の実行結果は次のように表示されます。各項目の見方は、表示例の後で説明します。

```
==================================================================
  MERIDIAN SCOREBOARD  ·  claude-sonnet-5  ·  5 trials/ticket
==================================================================

  Ticket T-4471   routing: account (x4); account, billing (x1)
    sso_addressed          5/5  #####
    billing_resolved       1/5  #....
    no_false_resolution    1/5  #....
    no_overclaim           3/5  ###..
    --> RESOLVED in 1/5 trials

------------------------------------------------------------------
  RESOLVED  1/5 trials
  COST      $0.55 total  ·  $0.11/trial
------------------------------------------------------------------
```

問い合わせ元の顧客が挙げた問題がすべて実際に処理された場合にのみ、その試行は解決（RESOLVED）として数えられます。意図的に高い基準を設けています。下に並ぶ4つのチェックはその内訳で、どの部分がスコアを押し下げているかを示します。`routing` には、コーディネーターが各試行で下した振り分けの判断が表示されます。同じチケットが5回の試行で異なる振り分けになっている場合、そのばらつきこそが、1回の実行では判断できない理由です。必ず割合で判断してください。

**所要時間に関する注意**：スコアボードはチケットを5回続けて処理するため、1回の実行に1〜2分かかります。完了してから結果をご確認ください。

作業のサイクル：実行する、割合を確認する、原因を調査する、1か所だけ変更する、再実行する。常に割合に注目してください。

---

**Partner Basecamp参加者向けの資料です。** 研修資料としての複製・再配布はご遠慮ください。ここで扱うパターンは、皆さまのお客様案件で自由にご活用いただけます。

## セットアップ：Claudeへの接続

最初に次のセルを実行してください。セットアップセルは、初回実行時に **`.env` ファイル**を作成します（gitignoreの対象のため、キーがコミットされることはありません）。ファイルを開き、`ANTHROPIC_API_KEY=` の後ろにキーを貼り付けて保存し、セルを再実行してください。キーはカーネルを再起動しても保持されるため、貼り付けは1回で済みます。*（`.env` がまだない場合は、代わりに、入力内容が伏せ字になる入力欄が表示されます。）* 緑色の **「✓ APIキーを確認しました」** バナーが表示されれば、接続は完了です。赤いバナーが表示された場合は、表示された指示に従ってセルを再実行してください。

これらのセルは、単なる `python` ではなく `{sys.executable}`（ノートブックのカーネル自身のPython）を実行します。これにより、スクリプトは常に、カーネルが `anthropic` をインストールした環境と同じ環境で動作します。下のセルが使用する `sys` もこのセットアップセルで読み込むため、必ず最初に実行してください。

In [ ]:
# このカーネルに依存パッケージをインストール。再実行しても安全で、制限付きのPython（PEP 668）でも動作する
import importlib.util, os, subprocess, sys

# ── 環境ガード ────────────────────────────────────────────────────────────
# 下のインストール処理と同じセルに置き、スキップできないようにしている。これにより、
# 素のシステムPythonにパッケージがインストールされることはない（従来の最悪のケースは、
# IT部門が管理するマシンでの --break-system-packages の実行）。ここで停止した場合は、
# SETUP.mdの「ノートブックが停止する理由」を参照。
def _in_isolated_env():
    """実行中のカーネルがvenv／virtualenv／conda環境、またはColabであればTrueを返す。
    判定はインタープリター自身（sys.*）に基づく。起動元のシェルから引き継いだ
    有効化用の環境変数は、sys.executableが実際にその環境の内部にある場合に
    限り信頼する。有効化済みのターミナルから起動したシステムPythonの
    カーネルもVIRTUAL_ENVを引き継ぐが、この場合は
    判定を通してはならない。"""
    if "google.colab" in sys.modules:
        return True  # Colabは使い捨ての独自ランタイムでサンドボックス化されている
    if sys.prefix != getattr(sys, "base_prefix", sys.prefix):
        return True  # PEP 405のvenv（python -m venv）。大半のconda環境も該当
    if hasattr(sys, "real_prefix"):
        return True  # 従来のvirtualenv
    exe = os.path.realpath(sys.executable)
    for var in ("VIRTUAL_ENV", "CONDA_PREFIX"):
        root = os.environ.get(var)
        if not root or not exe.startswith(os.path.realpath(root) + os.sep):
            continue  # シェルから引き継いだ情報にすぎず、カーネルは別の場所にある
        if var == "CONDA_PREFIX" and os.environ.get("CONDA_DEFAULT_ENV", "base") == "base":
            continue  # condaの共有 `base` は分離環境とみなさない
        return True
    return False

if os.environ.get("BASECAMP_ALLOW_SYSTEM_PYTHON") == "1":
    print("⚠️  環境ガードを回避しました（BASECAMP_ALLOW_SYSTEM_PYTHON=1）。"
          "意図的にこのPythonへインストールします。")
elif not _in_isolated_env():
    _HEAD = "✗ システムPythonを検出しました。何もインストールせずに停止しました"
    _BODY = (
        "ここにパッケージをインストールすると、マシン全体のPythonが変更されます。\n"
        "会社が管理するノートPCでは、IT部門への申請が必要になる場合があります。\n"
        "\n"
        "対処方法（2ステップ）：\n"
        "  1. ターミナルで、リポジトリのルートから次を実行してください：\n"
        "       python3 -m venv .venv\n"
        "       source .venv/bin/activate          # Windows: .venv\\Scripts\\activate\n"
        "       pip install -r requirements.txt\n"
        "  2. VS Codeで、右上のカーネル名をクリック → Select Another Kernel →\n"
        "     Python Environments → 末尾が.venvの環境を選択 → このセルを再実行してください。\n"
        "\n"
        "condaをご利用の場合は、`conda activate <env>`（`base` 以外）を実行してから、そのカーネルを選択してください。\n"
        "ご自身で管理するマシンを使うファシリテーターは、BASECAMP_ALLOW_SYSTEM_PYTHON=1 で回避できます。"
    )
    _shown = False
    try:  # 下のAPIキー確認と同じバナー表示（緑のボックス、エラー時は赤）
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fdecea;'
                'border:1.5px solid #b42318;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#b42318;font-weight:600;">' + _html.escape(_HEAD) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(_BODY) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    raise SystemExit(
        "環境ガードによりこのセルを停止しました。上のメッセージをご確認ください。"
        if _shown else "\n  " + _HEAD + "\n\n" + _BODY + "\n"
    )
else:
    print("✓ 環境ガード：分離されたインタープリターを検出しました。安全にインストールできます")
# ──────────────────────────────────────────────────────────────────────────


def _ensure_packages(requirements):
    """requirements: (import_name, pip_spec) のリスト。不足しているものだけを、
    実行中のインタープリターにインストールする。通常のインストール、ユーザー領域、
    PEP 668の上書き（ユーザー領域を優先し、システム全体は最後の手段）の順に試行する。
    いずれの試行も出力を表示しない（pipの出力はストリーム表示せずに取り込む）。そのため、
    制限付きのPython（HomebrewやDebian、PEP 668）でフォールバックが実際に成功した場合に、
    'externally-managed-environment' の長いエラー文が表示されることはない。
    すべての方法が失敗した場合に限り、生のトレースバックではなく、
    venvによる対処方法とともに原因を表示する。"""
    missing = [pip for mod, pip in requirements if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print("Installing " + ", ".join(missing) + "（初回のみ）。しばらくお待ちください…", flush=True)
    base = [sys.executable, "-m", "pip", "install", "-q"]
    last = None
    for extra in ([], ["--user"], ["--user", "--break-system-packages"], ["--break-system-packages"]):
        last = subprocess.run(base + extra + missing, capture_output=True, text=True)
        if last.returncode == 0:
            return
    pip_said = (last.stderr or last.stdout or "").strip().splitlines() if last else []
    tail = "\n      ".join(pip_said[-3:]) if pip_said else "（pipからの出力なし）"
    raise SystemExit(
        "\n  インストールできませんでした：" + ", ".join(missing) + "\n"
        "  このPythonは制限付き（PEP 668）か、オフラインの状態です。最も手早い対処方法はvenvの利用です：\n"
        f"      {sys.executable} -m venv .venv\n"
        "      source .venv/bin/activate          # Windowsの場合はSETUP.mdを参照\n"
        f"      pip install {' '.join(missing)}\n"
        "  その後、VS Code（右上のカーネル選択）で.venvのインタープリターを選択し、Run Allを実行してください。\n"
        "  社内プロキシやPyPIのブロックが原因の場合は、リポジトリのルートにあるSETUP.mdをご参照ください。\n"
        f"  (pip said: {tail})\n"
    )

_ensure_packages([("anthropic", "anthropic")])
print("✓ 依存パッケージの準備が完了しました")
import os
# 演習フォルダーにいることを確認する（リポジトリのルートでカーネルが起動した場合にも対応）
if not os.path.exists("Diagnose_Fix_Brief.py") and os.path.isdir("day1/04_diagnosing-ai-problems"):
    os.chdir("day1/04_diagnosing-ai-problems")

def _status(ok, msg):
    """ノートブックでは緑／赤のバナー、スクリプトとして実行した場合はプレーンテキストで表示する。"""
    try:
        from IPython import get_ipython
        shell = get_ipython()
        if shell is None or shell.__class__.__name__ != "ZMQInteractiveShell":
            raise RuntimeError("ノートブックのカーネル外のため、プレーンテキストのバナーを使用する")
        from IPython.display import display, HTML
        color = "#1a7f37" if ok else "#b42318"
        bg = "#e6f4ea" if ok else "#fdecea"
        icon = "✓" if ok else "✗"
        display(HTML(
            f'<div style="padding:12px 16px;border-radius:8px;background:{bg};'
            f'border:1.5px solid {color};color:{color};font-weight:600;'
            f'font-size:15px;font-family:sans-serif;">{icon} {msg}</div>'
        ))
    except Exception:
        print(("[OK] " if ok else "[!!] ") + msg)

import os
import pathlib

import anthropic

# ── Claudeへの接続：Anthropic APIまたはAmazon Bedrock ──
# どちらの認証情報でも動作する。種類はこのセルが自動で判別する
#   Anthropic API : ANTHROPIC_API_KEY=sk-ant-...
#   Amazon Bedrock：AWS_BEARER_TOKEN_BEDROCK=...  と  AWS_REGION=us-east-1
# 使用する認証情報を、このセルが作成する.envファイル（gitignoreの対象のため、コミットされない）に記入するか、
# シェルでexportする。シェルの値は.envファイルの値より優先される。
_ENV_TEMPLATE = (
    "# Anthropic API key — paste after the = (no quotes, no spaces), then save and\n"
    "# re-run the setup cell. Get one at https://console.anthropic.com/\n"
    "ANTHROPIC_API_KEY=paste-your-key-here\n"
    "\n"
    "# --- Using Amazon Bedrock instead? Comment out the line above and fill these in:\n"
    "# AWS_BEARER_TOKEN_BEDROCK=paste-your-bedrock-api-key-here\n"
    "# AWS_REGION=us-east-1\n"
)


def _resolve_env_file():
    """作業ディレクトリから上位へたどり、最も近い既存の.envを探す（ルートに.envを1つ置けば
    すべての演習で共用できる）。まだ存在しない場合はリポジトリのルートを指し、
    ノートブックを単独で開いた場合はこのフォルダーを指す。"""
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / ".env").is_file():
            return d / ".env"
    root = next((d for d in [here, *here.parents]
                 if (d / "SETUP.md").exists() or (d / ".git").exists()), here)
    return root / ".env"


_env_file = _resolve_env_file()
if not _env_file.exists():
    _env_file.write_text(_ENV_TEMPLATE)
    print(f"Created {_env_file.name} in {_env_file.parent} — open it, add your key, "
          "save, then re-run this cell.")

# 簡易的な.envパーサー（python-dotenvに依存しない）。実行のたびに読み直すため、キーを貼り付けて
# 再実行すれば反映される。環境（シェル／Claude Code／CI）に実際の値がある場合は
# そちらが優先され、プレースホルダーが残ることはない。
_file = {}
for _line in (_env_file.read_text().splitlines() if _env_file.exists() else []):
    _line = _line.strip()
    if _line and not _line.startswith("#") and "=" in _line:
        _k, _v = _line.split("=", 1)
        _file[_k.strip()] = _v.strip().strip('"').strip("'")
for _k, _v in _file.items():
    if _k != "ANTHROPIC_API_KEY":
        os.environ.setdefault(_k, _v)

_shell_key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
_anthropic_key = _shell_key if _shell_key.startswith("sk-ant-") else _file.get("ANTHROPIC_API_KEY", "").strip()
_bedrock_token = os.environ.get("AWS_BEARER_TOKEN_BEDROCK", "").strip()
_bedrock_region = os.environ.get("AWS_REGION", "").strip()

if _anthropic_key.startswith("sk-ant-"):
    PROVIDER = "anthropic"
elif _bedrock_token:
    PROVIDER = "bedrock"
else:
    PROVIDER = None


def _needs_credentials(head, body):
    """黄色の警告バナーを表示して停止する。数セル先ではなく、この時点でセットアップを失敗させるため。"""
    _shown = False
    try:
        from IPython import get_ipython
        if get_ipython().__class__.__name__ == "ZMQInteractiveShell":
            import html as _html
            from IPython.display import HTML, display
            display(HTML(
                '<div style="padding:12px 16px;border-radius:8px;background:#fff8c5;'
                'border:1.5px solid #9a6700;font-size:15px;font-family:sans-serif;">'
                '<div style="color:#9a6700;font-weight:600;">' + _html.escape(head) + '</div>'
                '<pre style="margin:10px 0 0;font-family:inherit;font-size:14px;font-weight:400;'
                'color:#141413;white-space:pre-wrap;">' + _html.escape(body) + '</pre></div>'
            ))
            _shown = True
    except Exception:
        pass
    if not _shown:
        print("\n" + head + ":\n   " + body.replace("\n", "\n   ") + "\n")
    raise SystemExit("認証情報が設定されていません。上のメッセージをご確認ください。")


if PROVIDER is None:
    _needs_credentials(
        "📋 続行するには認証情報を追加してください",
        f"Open this file:  {_env_file}\n"
        "\n"
        "Anthropic APIをご利用の場合は、次を設定してください：\n"
        "    ANTHROPIC_API_KEY=sk-ant-...\n"
        "\n"
        "Amazon Bedrockをご利用の場合は、次の両方を設定してください：\n"
        "    AWS_BEARER_TOKEN_BEDROCK=<your Bedrock API key>\n"
        "    AWS_REGION=us-east-1          # モデルを有効化しているリージョン\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )

if PROVIDER == "bedrock" and not _bedrock_region:
    _needs_credentials(
        "📋 Bedrockにはリージョンの指定が必要です",
        f"Found AWS_BEARER_TOKEN_BEDROCK but no AWS_REGION.\n"
        f"\n"
        f"Open this file:  {_env_file}\n"
        "Bedrockのモデルを有効化しているリージョンを追記してください。例：\n"
        "    AWS_REGION=us-east-1\n"
        "\n"
        "ファイルを保存してから、このセルの▶をもう一度クリックしてください。"
    )


def _model(name):
    """BedrockのモデルIDには `anthropic.` というプレフィックスが付きます。Anthropic APIでは、プレフィックスなしのIDを使用します。"""
    return f"anthropic.{name}" if PROVIDER == "bedrock" else name


# この演習で使用するモデル。利用中のプロバイダーに合わせてIDを解決する
MODEL = _model("claude-sonnet-5")        # この演習の主力モデル
FAST_MODEL = _model("claude-haiku-4-5")  # 低コストかつ高速（接続確認、ジャッジ用）
BIG_MODEL = _model("claude-opus-4-8")    # より大きなモデルを試す場合


def _make_client(timeout, max_retries=2):
    if PROVIDER == "bedrock":
        from anthropic import AnthropicBedrockMantle
        return AnthropicBedrockMantle(aws_region=_bedrock_region,
                                      timeout=timeout, max_retries=max_retries)
    return anthropic.Anthropic(api_key=_anthropic_key,
                               timeout=timeout, max_retries=max_retries)


# 接続確認：認証情報に加え、このモデルを実際に利用できるかも検証する
# Bedrockでは、キーが有効でも、アカウント／リージョンでモデルが有効化されていなければ404になることがある。
# そのため、認証情報の形式を確認するだけでなく、実際のモデルIDにpingを送る。
_probe = _make_client(timeout=30.0, max_retries=1)
try:
    _probe.messages.create(model=FAST_MODEL, max_tokens=1,
                           messages=[{"role": "user", "content": "ping"}])
except anthropic.NotFoundError:
    if PROVIDER == "bedrock":
        _status(False, f"Bedrock reached, but model '{FAST_MODEL}' isn't available to you in "
                       f"{_bedrock_region}. Enable model access for it in the Bedrock console "
                       f"(or switch AWS_REGION to a region where it is enabled), then re-run.")
    else:
        _status(False, f"Model '{FAST_MODEL}' not found for this key.")
    raise SystemExit("モデルを利用できません。上のメッセージをご確認ください。")
except (anthropic.AuthenticationError, anthropic.PermissionDeniedError):
    if PROVIDER == "bedrock":
        _status(False, "Bedrockキーが拒否されました。AWS_BEARER_TOKEN_BEDROCKの値と、"
                       "Bedrockの呼び出し権限が付与されていることを確認してから、このセルを再実行してください。")
    else:
        _status(False, "キーが拒否されました。このセルを再実行し、キー全体を貼り付けてください"
                       "（sk-ant-で始まります）。")
    raise SystemExit("認証情報が受け付けられませんでした。このセルを再実行して、もう一度お試しください。")
except Exception as exc:
    _status(False, "APIに接続できませんでした（" + type(exc).__name__ + "）。"
                   "ネットワーク接続を確認してから、このセルを再実行してください。")
    raise
else:
    if PROVIDER == "anthropic":
        os.environ["ANTHROPIC_API_KEY"] = _anthropic_key  # 以降のセルや !python がこの値を参照する
        _status(True, "APIキーを確認しました。Claudeに接続されています。")
    else:
        _status(True, f"Bedrock key verified ({_bedrock_region}) - you're connected to Claude "
                      f"as {MODEL}.")

# 実際に使用するクライアント。非ストリーミング呼び出しで max_tokens>21333 を使うには、長めのタイムアウトが必要
client = _make_client(timeout=900.0)


## 1. 状況の把握
次のセルを実行すると、Priyaのメールがこの場に表示されます。受信トレイに届いたばかりのつもりでお読みください。Priyaのチームは3週間前にサポートエージェントをリリースしましたが、その信頼性に疑問を抱き始めています。経営陣から繰り返し問われているのは、モデルの選定が誤りだったのかという点です。Priyaが求めているのは、実際の問題は何か、修正で対応できるのか、経営陣に示せる根拠は何か、という3点への率直な回答です。

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("Priya_Email.md").read().split("---", 1)[-1]))

## 2. ベースラインの解決率の測定
次のセルを実行してください。チケットT-4471のRESOLVEDの割合とコストが表示されます。この割合が、Priyaが懸念しているパイロットの現状であり、皆さまの出発点となります。数値を記録しておいてください。スコアボードの実行結果はすべて `runs.jsonl` に記録され、スコアボードの下にRUN HISTORYブロックとして再表示されるため、ベースラインは演習終了まで画面上で確認できます。（チケットを5回処理するため、1〜2分かかります。各試行の結果は完了するたびに表示されます。）

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

## 3. 「モデルが原因」という仮説の検証
Priyaは、モデルの性能不足が原因だと考えています。同じように考えている方もいらっしゃるでしょう。そこで、この仮説を正面から検証します。次のセルは、同じ評価を、より大規模で高価なモデルで実行します。新たな割合とコストが表示されます。そのうえで、次の2点をご検討ください。大規模なモデルは実際にチケットを解決できたのか。そして、それを確認するためにどれだけのコストがかかったのか。ここで得られる結論は、皆さまご自身の検証から導かれたものです。

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5 --model claude-opus-4-8

## 4. チケット1件の処理過程の観察
割合からは、失敗していることは分かりますが、どのように失敗しているかは分かりません。そこで、1回の実行を最初から最後まで観察します。コーディネーターのすべての動作が表示されます。何を調べたか、どのスペシャリストにチケットを引き渡したか、お客様にどのような返信を送ったか。これをメールの内容と照らし合わせてください。お客様は複数の問題を挙げていました。エージェントはそのうちいくつに対応し、どこで処理を止めたでしょうか。

表示内容の読み方が分からない場合は、トランスクリプトをClaudeに貼り付け、考えを言葉にしながら一緒に読み解いてください。*「これは、お客様が未解決だとしているチケットをクローズしたサポートエージェントのトレースです。エージェントが何をしたのか、どこで誤った可能性があるのかを順に説明してください。」* Claudeの回答はそのまま結論とせず、検証すべき手がかりとして扱ってください。

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --ticket T-4471

## 5. エージェント定義の確認
失敗の原因について仮説が立ちました。次は、その挙動を生んだシステムを確認します。「エージェント」の実体はコードではなく、このフォルダーにあるテキストファイルで、このエディタで直接開けます。

まず、コーディネーターの `system-prompt-coordinator.txt` から確認してください。各チケットの担当を決めているのはこのファイルです。チケットの分類と引き渡しについて、どのように指示されているかをお読みください。その後は、ツール一覧（`coordinator-tools.json`）や各スペシャリストのファイルも自由にご確認いただけます。

探すのは、先ほど観察した挙動の原因となっている指示や構成です。そこが書き換えるべき箇所になります。行き詰まった場合は、Claudeに質問してください。*「これはコーディネーターへの指示です。2つの別々の問題を含むチケットで、一方しか処理されませんでした。この指示のどの部分が原因として考えられますか？」*

## 6. 1か所の変更と再測定
該当箇所は見つかりましたか？　変更してファイルを保存し、次のセルを実行してください。割合は変化したでしょうか。スコアボードの下のRUN HISTORYブロックには、ベースラインを含むこれまでのすべての実行結果が表示されるため、変更前後をその場で比較できます。変化がなければ、仮説が誤っていたか、部分的にしか正しくなかったと考えられます。トレースに戻ってください。割合が上がった場合は、有効な調整項目を特定できたことになります。エージェントが毎回確実にチケットを解決できるようになるまで、調整を続けてください。モデルは最後まで変更しません。

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --trials 5

## 7. ホールドアウトでの確認
1件のチケットであれば、偶然に解決してしまうこともあります。次のセルは、これまで使用していないホールドアウトのチケットを実行します。お客様も問題も異なります。これらのチケットも解決できれば、修正は有効と判断できます。解決できない場合は、T-4471に合わせた調整にとどまっており、さらに改善が必要です。

In [ ]:
!{sys.executable} Diagnose_Fix_Brief.py --holdout --trials 3

## 8. Priya宛ての回答の作成
これでPriyaに回答できます。`client-brief-template.md` に、次の内容を記入してください。実際に何が問題を引き起こしていたのか（平易な言葉で）、何を変更したか、その実証結果（変更前後の割合とコスト）、そしてPriyaから必ず寄せられる質問 *「より高性能なモデルに費用をかければよいのではないか」* への回答です。この検証は手順3で実施済みのため、答えはすでにお持ちのはずです。